# Evaluate SAM FathomNet output
Read ground truth annotations from FathomNet in COCO format. Compare to SAMv1 outputs

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
import os 
from PIL import Image
import cv2
import pylab
from sklearn import metrics
import glob, os
#pylab.rcParams['figure.figsize'] = (16.0, 20.0)

### Load datasets

In [ ]:
gt = COCO('fathomnet-batho/batho-inner-dataset.json')
sam = COCO('fathomnet-batho/sam-procd-inner-filt-scores.json')

Skip the three images that were prompted with the outer rather then the inner filter

In [ ]:
skip = ['9d00b67d-b88e-429b-86c5-1c4d1a480906.png',
        '898d7773-9cdd-4017-b797-3e028b83e969.png', 
        'b62bbe68-c908-420d-9773-40ad840ea303.png'
    ]

imgIds = [line for line in sam.getImgIds() if sam.loadImgs(line)[0]['file_name'] not in skip]

### Compute mean IoU 

Compute the mean intersection over union between SAM output and ground truth 

In [ ]:
# Create evaluation object
eval = COCOeval(gt, sam, 'bbox')
eval.evaluate()

# compute IoU
out = []
for ii in imgIds:
    test = eval.computeIoU(ii, 1)

    out.extend(np.amax(test, axis=1))  # sam anns (some zeros where there is original )

out = np.asarray(out)

print(f'med iou: {np.median(out)}')
print(f'mean iou: {np.mean(out)}')
print(f'max iou: {np.max(out)}')

# make a plot
counts, bins = np.histogram(out)

fig, ax = plt.subplots()
_ = ax.hist(bins[:-1], bins, weights=counts)
ax.set_xlabel('Intersection over Union', fontsize=16)
ax.set_ylabel('Counts', fontsize=16)
ax.set_xlim([0, 1])
ax.tick_params(labelsize=14)
fig.tight_layout()

### Compute distance

Get the distance between the proposed and ground truth bounding box center.

In [ ]:
out = []
for ii in imgIds:

    # make list of groundtruth bbox centers
    gt_cen = []
    for ann in gt.loadAnns(gt.getAnnIds(imgIds=ii, catIds=1)):
        bbox = ann['bbox']
        center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
        gt_cen.append(center)


    # make list of SAM bbox centers
    sam_cen = []
    for ann in sam.loadAnns(sam.getAnnIds(imgIds=ii, catIds=1)):
        bbox = ann['bbox']
        center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
        sam_cen.append(center)

    zz = metrics.pairwise.euclidean_distances(np.array(gt_cen), np.array(sam_cen))

    out.extend(np.amin(zz, axis=1).tolist())

out = np.asarray(out)

print(f'med dist: {np.median(out)}')
print(f'mean dist: {np.mean(out)}')
print(f'max dist: {np.max(out)}')

# make a plot
counts, bins = np.histogram(out)

fig, ax = plt.subplots()
_ = ax.hist(bins[:-1], bins, weights=counts)
ax.set_xlabel('Euclidean distance (pixels)', fontsize=16)
ax.set_ylabel('Counts', fontsize=16)
ax.set_xlim([0, 40])
ax.tick_params(labelsize=14)
fig.tight_layout()

### L and W, SAM v. gt

Dimensions of SAM outputs compared to ground truth annotations.

In [ ]:
gt_df = pd.DataFrame(gt.loadAnns(gt.getAnnIds(imgIds=imgIds)))
gt_df[['x', 'y', 'w', 'h']] = pd.DataFrame(gt_df['bbox'].to_list(), index=gt_df.index)

sam_df = pd.DataFrame(sam.loadAnns(sam.getAnnIds(imgIds=imgIds)))
sam_df[['x', 'y', 'w', 'h']] = pd.DataFrame(sam_df['bbox'].to_list(), index=sam_df.index)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(16,8))
sam_df[sam_df['category_id']==1]['w'].hist(ax=ax[0])
sam_df[sam_df['category_id']==1]['h'].hist(ax=ax[0])
ax[0].legend(['width', 'length'])
ax[0].set_title('SAM output')


gt_df[gt_df['category_id']==1]['w'].hist(ax=ax[1])
gt_df[gt_df['category_id']==1]['h'].hist(ax=ax[1])
ax[1].legend(['width', 'length'])
ax[1].set_title('Ground truth')

fig.supxlabel('Pixels')
fig.supylabel('Count')
plt.tight_layout()

In [ ]:
print(f"median SAM bbox: {sam_df[sam_df['category_id']==1]['w'].median()} x {sam_df[sam_df['category_id']==1]['h'].median()}")
print(f"median gt bbox: {gt_df[gt_df['category_id']==1]['w'].median()} x {gt_df[gt_df['category_id']==1]['h'].median()}")

### Plot individual frame

Image plotting with both SAM and ground truch bboxes.

In [ ]:
# Select an image to visualize, assuming images have been downloaded locally

ii = 92
image = cv2.imread(f"fathomnet-batho/images/{sam.imgs[ii]['file_name']}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# compare all annotations 

# make list of groundtruth bbox centers
gt_cen = []
for ann in gt.loadAnns(gt.getAnnIds(imgIds=ii, catIds=1)):
    bbox = ann['bbox']
    center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
    gt_cen.append(center)

# make list of SAM bbox centers
sam_cen = []
for ann in sam.loadAnns(sam.getAnnIds(imgIds=ii, catIds=1)):
    bbox = ann['bbox']
    center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
    sam_cen.append(center)

# plot the center of SAM bounding boxes
for cen in sam_cen:
    cv2.circle(image, (int(cen[0]), int(cen[1])), 7, (255, 127, 14), cv2.FILLED)

# plot the center of ground truth bboces
for cen in gt_cen:
    cv2.circle(image, (int(cen[0]), int(cen[1])), 7, (0, 255, 0), cv2.FILLED)

fig, ax = plt.subplots(figsize=(16,20))
ax.imshow(image); ax.axis('off')
annIds = gt.getAnnIds(imgIds=ii)
anns = gt.loadAnns(annIds)

# plot gt
polygons = []
for ann in anns:
    [bbox_x, bbox_y, bbox_w, bbox_h] = ann['bbox']
    poly = [[bbox_x, bbox_y], [bbox_x, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y]]
    np_poly = np.array(poly).reshape((4,2))
    polygons.append(Polygon(np_poly))

c = [0.0, 1.0, 0.0]
p = PatchCollection(polygons, facecolor='none', linewidths=0, alpha=0.4)
ax.add_collection(p)
p = PatchCollection(polygons, facecolor='none', edgecolors=c, linewidths=2)
ax.add_collection(p)

# plot the sam proposals
annIds = sam.getAnnIds(imgIds=ii, catIds=1)
anns = sam.loadAnns(annIds)
polygons = []
for ann in anns:
    [bbox_x, bbox_y, bbox_w, bbox_h] = ann['bbox']
    poly = [[bbox_x, bbox_y], [bbox_x, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y]]
    np_poly = np.array(poly).reshape((4,2))
    polygons.append(Polygon(np_poly))

c = [1.0, 0.4980392156862745, 0.054901960784313725]
p = PatchCollection(polygons, facecolor='none', linewidths=0, alpha=0.4)
ax.add_collection(p)
p = PatchCollection(polygons, facecolor='none', edgecolors=c, linewidths=2)
ax.add_collection(p)

### Plot individual frame with seg

Image plotting with original bboxes and SAM segmentation masks

In [ ]:
ii = 92
image = cv2.imread(f"fathomnet-batho/images/{sam.imgs[ii]['file_name']}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# compare all annotations 

# make list of groundtruth bbox centers
gt_cen = []
for ann in gt.loadAnns(gt.getAnnIds(imgIds=ii, catIds=1)):
    bbox = ann['bbox']
    center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
    gt_cen.append(center)

# plot the center of ground truth bboces
for cen in gt_cen:
    cv2.circle(image, (int(cen[0]), int(cen[1])), 7, (0, 255, 0), cv2.FILLED)

# plot the segmentation masks
for ann in sam.loadAnns(sam.getAnnIds(imgIds=ii)):
    poly = np.array(ann['segmentation'])
    cv2.drawContours(image, [poly], 0, (255, 127, 14), 2)

fig, ax = plt.subplots(figsize=(16,20))
ax.imshow(image); ax.axis('off')

annIds = gt.getAnnIds(imgIds=ii)
anns = gt.loadAnns(annIds)

# plot gt
polygons = []
for ann in anns:
    [bbox_x, bbox_y, bbox_w, bbox_h] = ann['bbox']
    poly = [[bbox_x, bbox_y], [bbox_x, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y]]
    np_poly = np.array(poly).reshape((4,2))
    polygons.append(Polygon(np_poly))
    #color.append(c)

c = [0.0, 1.0, 0.0]
p = PatchCollection(polygons, facecolor='none', linewidths=0, alpha=0.4)
ax.add_collection(p)
p = PatchCollection(polygons, facecolor='none', edgecolors=c, linewidths=2)
ax.add_collection(p)